In [ ]:
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
import re
import json
import re
import sys
import os
import numpy as np
from glob import glob
from typing import List
from copy import deepcopy
import numpy as np
sys.path.append('../data_ops')
sys.path.append('../')
import torch
# Initialize OpenAI client
client = OpenAI(api_key="api-key")

# Load datasets
opentom_dataset = load_dataset("OpenToM/opentom")  # Replace with the correct Hugging Face dataset ID
opentom_long_dataset = load_dataset("OpenToM/opentom_long")

# Load metadata for detailed question types
with open("opentom_data/metadata.json", "r") as file:
    opentom_metadata = json.load(file)
    
with open("opentom_data/metadata_long.json", "r") as file:
    opentom_long_metadata = json.load(file)

# Combine datasets into a single dataframe
opentom_df = pd.DataFrame(opentom_dataset["train"])
opentom_long_df = pd.DataFrame(opentom_long_dataset["train"])
combined_df = pd.concat([opentom_df, opentom_long_df], ignore_index=True)

# Filter if needed
#selected_questions = combined_df[combined_df["genre"] == "location_fg_fo"].head(100)
selected_questions = combined_df
# Placeholder for responses and correctness
responses = []
correctness = []

# Function to parse AI answers
def parse_ai_answer(response_content):
    match = re.search(r"(?:the correct answer is:\s*[A-Z]\.\s*)(.+?)(?:\.|$)", response_content, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return response_content.strip()

# Process selected questions
for index, row in selected_questions.iterrows():
    question = row["question"]
    narrative = row["narrative"]
    correct_answer = row["answer"]
    metadata = opentom_metadata.get(str(row["id"]), {})  # Access specific metadata by question ID
    preferences = metadata.get("preferences", {})
    personality = metadata.get("personality", {})
    sentiment_statement = metadata.get("sentiment_statement", "")
    intention = metadata.get("intention", "")
    
    # Construct prompt
    prompt = (
        f"Read the following narrative and answer the question based on the details provided.\n\n"
        f"Narrative: {narrative}\n\n"
        f"Question: {question}\n\n"
        f"Note: Consider these details:\n"
        f"Preferences: {preferences}\n"
        f"Personality: {personality}\n"
        f"Sentiment Statement: {sentiment_statement}\n"
        f"Intention: {intention}\n\n"
        f"Please explain your reasoning after providing the answer."
    )
    
    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    # Extract and evaluate response
    raw_ai_answer = response.choices[0].message.content
    #ai_answer = parse_ai_answer(raw_ai_answer)
    responses.append(raw_ai_answer)
    #correctness.append(is_correct)

# Add results to the dataframe
selected_questions["ai_answer"] = responses
selected_questions["is_correct"] = None

# Calculate accuracy
accuracy = sum(correctness) / len(correctness) * 100
print(f"Total Accuracy: {accuracy:.2f}%")

# Save results to CSV
output_file = "opentom_responses_with_accuracy.csv"
selected_questions.to_csv(output_file, index=False)
print(f"Responses with accuracy saved to {output_file}")



class OpenToMEvaluator():

    def __init__(self) -> None:
        self.datautils = DataUtils()
        self.baseline_labels = BaselineLabels()
        self.opentom_utils = OpenToMUtils()
        # self._init_sentiment_model()

    @staticmethod 
    def remove_determinant(word: str) -> str:
        determinants = ['a', 'an', 'the']
        for det in determinants:
            if word.startswith(det):
                return word[len(det):].strip()
        return word

    @staticmethod
    def compute_lexical_overlap(pred: str, location: str) -> float:
        pred = pred.lower().replace('_', ' ').replace("'s", '')
        location = location.lower().replace('_', ' ').replace("'s", '')
        score = 0 
        pred = pred.replace('.', '').split()
        location = location.split()
        visited_word = []

        for word in pred:
            if word in location and word not in visited_word:
                score += 1
                visited_word.append(word)

        return score / len(location)

    @staticmethod
    def parse_cot_answer(answer: str) -> str:
        # cot typically generate answer in the last sentence or paragraph
        if '\n' in answer:
            answer = answer.split('\n')[-1]
        else:
            answer = answer.split('Therefore')[-1]
        return answer
    
    def check_answer_for_fg_location(self, prediction: str, answer: str, original_place: str, move_to_place: str) -> list:

        # truncate prediction as some of them contain explanations
        answer = self.remove_determinant(answer).lower()

        original_place = self.remove_determinant(original_place).lower()
        move_to_place = self.remove_determinant(move_to_place).lower()

        gt_label, pred_label = None, None

        original_place_score = self.compute_lexical_overlap(prediction, original_place)
        move_to_place_score = self.compute_lexical_overlap(prediction, move_to_place)

        if original_place_score == move_to_place_score:
            pred_label = 3

        if original_place_score > move_to_place_score:
            pred_label = 1
        elif original_place_score < move_to_place_score:
            pred_label = 2

        if original_place == answer:
            gt_label = 1 
        elif move_to_place == answer:
            gt_label = 2

        return [gt_label, pred_label]

    @staticmethod
    def check_answer_for_cg_location(prediction: str, answer: str) -> list:
        prediction = prediction.lower()
        answer = answer.lower()

        if 'no' in prediction and 'yes' not in prediction:
            pred_label = 0
        elif 'yes' in prediction and 'no' not in prediction:
            pred_label = 1
        else:
            pred_label = -1

        if 'no' in answer:
            gt_label = 0 
        elif 'yes' in answer:
            gt_label = 1

        return [gt_label, pred_label]

    def check_fullness_answer(self, prediction: str, answer: str) -> list:

        prediction = prediction.replace('.', '').lower()

        less_full_answer_list = ['less full', 'emptier', 'more empty']
        more_full_answer_list = ['more full', 'fuller']

        pred_label, gt_label = None, None
        for less_full_ans in less_full_answer_list:
            if less_full_ans in prediction:
                pred_label = 1

        if not pred_label:
            for more_full_ans in more_full_answer_list:
                if more_full_ans in prediction:
                    pred_label = 2

        if not pred_label:
            if "equally full" in prediction:
                pred_label = 3

        if not pred_label:
            pred_label = -1  # corrupted

        if answer == 'less full':
            gt_label = 1 
        elif answer == 'more full':
            gt_label = 2
        elif answer == 'equally full':
            gt_label = 3

        return [gt_label, pred_label]

    def check_accessibility_answer(self, prediction: str, answer: str) -> list:

        prediction = prediction.replace('.', '').lower()

        pred_label, gt_label = None, None
        if "more accessible" in prediction:
            pred_label = 1
        elif "less accessible" in prediction:
            pred_label = 2
        elif "equally accessible" in prediction:
            pred_label = 3
        else:
            pred_label = -1  # corrupted

        if answer == 'more accessible':
            gt_label = 1 
        elif answer == 'less accessible':
            gt_label = 2
        else:
            gt_label = 3

        return [gt_label, pred_label]

    def check_attitude_answer(self, prediction: str, answer: str) -> list:

        prediction = prediction.lower()
        answer = answer.lower()

        answer_map = {
            'a': 'positive',
            'b': 'neutral',
            'c': 'negative'
        }
        prediction_token = prediction.split('\n\n')[-1].split(':')[-1].split('.')[0].strip().lower()

        gt_label, pred_label = None, None

        if answer == 'positive':
            gt_label = 1 
        elif answer == 'negative':
            gt_label = 2
        else:
            gt_label = 3

        try:
            prediction = answer_map[prediction_token]

            if prediction == 'positive':
                pred_label = 1 
            elif prediction == 'negative':
                pred_label = 2
            else:
                pred_label = 3

        except:
            if 'positive' in prediction_token and 'negative' in prediction_token:
                pred_label = -1
            elif 'positive' in prediction_token and 'neutral' in prediction_token:
                pred_label = -1
            elif 'neutral' in prediction_token and 'negative' in prediction_token:
                pred_label = -1
            elif 'positive' in prediction_token:
                pred_label = 1 
            elif 'negative' in prediction_token:
                pred_label = 2 
            elif 'neutral' in prediction_token:
                pred_label = 3 
            else:
                pred_label = -1

        return [gt_label, pred_label]


    def evaluate(self, result_path: str, location_granularity: str, perspective: str) -> dict:

        result_data = self.datautils.load_json(result_path)
        meta_data =self.datautils.load_json('../data/opentom_data/meta_data.json')

        # added cot prompting for entity state questions
        cot_flag, llama_flag = False, False
        if 'cot' in result_path:
            cot_flag = True
        if 'llama' in result_path:
            llama_flag = True

        # NOTE: 
        # fo -> first order; so -> second order; 0h -> 0-hop 1h -> 1-hop esq -> entity state question
        location_fo, location_so = [[] for _ in range(5)], [[] for _ in range(5)]
        multihop_fo, multihop_so = [[] for _ in range(5)], [[] for _ in range(5)]
        attitude = [[] for _ in range(5)]

        for batch_num, batch_content in result_data.items():

            cur_batch_idx = int(batch_num.split('-')[-1]) - 1

            for key, val in batch_content.items():

                mover, affected_char, eoi, original_place, move_to_place = meta_data[key]['plot_info'].values()
                places = [original_place.replace('_', ' '), move_to_place.replace('_', ' ')]

                for question_id, question_dict in val.items():

                    cur_question_type = question_dict['type']
                    question_content = question_dict['question']

                    pred_answer = question_dict['prediction'].strip()
                    gt_answer = question_dict['answer'].strip()

                    # NOTE: evaluate based on the character
                    if perspective == 'observer':
                        if mover in question_content and affected_char not in question_content:
                            continue

                        if mover in question_content and affected_char in question_content:
                            question_tokens = question_content.replace("'s", '').replace(',', '').split()

                            mover_idx = question_tokens.index(mover)
                            affected_char_idx = question_tokens.index(affected_char)

                            if mover_idx < affected_char_idx:
                                continue

                    elif perspective == 'mover':
                        if mover not in question_content and affected_char in question_content:
                            continue

                        if mover in question_content and affected_char in question_content:
                            question_tokens = question_content.replace("'s", '').replace(',', '').split()

                            mover_idx = question_tokens.index(mover)
                            affected_char_idx = question_tokens.index(affected_char)

                            if mover_idx > affected_char_idx:
                                continue

                    if cot_flag:
                        pred_answer = self.parse_cot_answer(pred_answer)

                    if cur_question_type == 'location-fo':

                        if location_granularity == 'fine':
                            gt, pred = self.check_answer_for_fg_location(pred_answer, gt_answer, original_place, move_to_place)
                        else:
                            gt, pred = self.check_answer_for_cg_location(pred_answer, gt_answer)
                        
                        location_fo[cur_batch_idx].append(tuple(('location', gt, pred)))

                    elif cur_question_type == 'location-so':
                        if location_granularity == 'fine':
                            gt, pred = self.check_answer_for_fg_location(pred_answer, gt_answer, original_place, move_to_place)
                        else:
                            gt, pred = self.check_answer_for_cg_location(pred_answer, gt_answer)

                        location_so[cur_batch_idx].append(tuple(('location', gt, pred)))

                    elif cur_question_type == 'multihop-fo':

                        if 'fullness' in question_content:
                            gt, pred = self.check_fullness_answer(pred_answer, gt_answer)

                            multihop_fo[cur_batch_idx].append(tuple(('fullness', gt, pred)))

                        elif 'accessibility' in question_content:
                            if '|' in gt_answer:
                                gt_answer = "equally accessible"

                            if isinstance(gt_answer, list):
                                gt_answer = [ele for ele in gt_answer if ele != 'corrupted']
                                assert len(gt_answer) == 1, print(key, gt_answer)
                                gt_answer = gt_answer[0]

                            gt, pred = self.check_accessibility_answer(pred_answer, gt_answer)

                            multihop_fo[cur_batch_idx].append(tuple(('accessibility', gt, pred)))

                    elif cur_question_type == 'multihop-so':
                        if 'fullness' in question_content:
                            gt, pred = self.check_fullness_answer(pred_answer, gt_answer)

                            multihop_so[cur_batch_idx].append(tuple(('fullness', gt, pred)))

                        elif 'accessibility' in question_content:

                            if '|' in gt_answer:
                                gt_answer = "equally accessible"

                            if isinstance(gt_answer, list):
                                gt_answer = [ele for ele in gt_answer if ele != 'corrupted']
                                assert len(gt_answer) == 1 
                                gt_answer = gt_answer[0]

                            gt, pred = self.check_accessibility_answer(pred_answer, gt_answer)

                            multihop_so[cur_batch_idx].append(tuple(('accessibility', gt, pred)))


                    elif cur_question_type == 'attitude':
                        gt, pred = self.check_attitude_answer(pred_answer, gt_answer)

                        attitude[cur_batch_idx].append(tuple(('attitude', gt, pred)))

        result_dict = {
            'location-fo': location_fo,
            'location-so': location_so,
            'multihop-fo': multihop_fo,
            'multihop-so': multihop_so,
            'attitude': attitude,
        }

        return result_dict
    
    
    



import re
import os
import json
import yaml
import pickle


class DataUtils():

    @staticmethod
    def check_file_existence(fpath):
        '''
        check_file_existence function to check if file exists

        Args:
            fpath (str): path to the file

        Returns:
            bool: True if file exists, False otherwise
        '''
        if os.path.exists(fpath):
            raise Exception('File already tagged')

    @staticmethod
    def save_json(file: dict, fpath: str) -> None:
        '''
        save_json function to save dictionary as json file

        Args:
            file (dict): dictionary to be saved
            fpath (str): path to the json file
        '''
        with open(fpath, 'w') as f:
            json.dump(file, f, indent=4)
        f.close()

    @staticmethod
    def load_tomi(fpath: str) -> dict:
        '''
        load_txt function to load local txt file

        Args:
            fpath (str): path to the txt file

        Returns:
            dict: txt file as a dictionary
        '''
        with open(fpath, 'r') as f:
            raw_data = f.readlines()
        
        data = {}
        counter = 0
        for entry in raw_data:
            if entry.strip().split()[0] == '1':
                if counter != 0:
                    temp_entry = [e.strip() for e in temp_entry]
                    temp_content = '\n'.join(temp_entry[:-1])
                    temp_question, temp_answer = temp_entry[-1].split('?')
                    data[str(counter)] = {
                        'content': temp_content,
                        'question': temp_question.strip() + '?',
                        'answer': temp_answer.strip()
                    }
                counter += 1
                temp_entry = [entry]
            else:
                temp_entry.append(entry.strip())

        return data

    @staticmethod
    def load_txt(fpath: str) -> str:
        '''
        load_txt function to load local txt file

        Args:
            fpath (str): path to the txt file

        Returns:
            str: txt file as a string
        '''
        with open(fpath, 'r') as f:
            data = f.read()
        f.close()

        return data

    @staticmethod
    def load_json(fpath: str) -> dict:
        '''
        load_json function to load json file

        Args:
            fpath (str): path to the json file

        Returns:
            dict: json file as a dictionary
        '''
        with open(fpath, 'r') as f:
            data = json.load(f)
        f.close()

        return data

    @staticmethod
    def load_jsonl(fpath: str) -> list:
        '''
        load_jsonl function to load jsonl file

        Args:
            fpath (str): path to the jsonl file

        Returns:
            list: jsonl file loaded as list of dictionaries
        '''

        new_data = []

        with open(fpath, 'r') as f:
            raw_data = f.readlines()
            for line in raw_data:
                cur_line = json.loads(line)
                new_data.append(cur_line)

        return new_data
    
    @staticmethod
    def load_yaml(fpath: str) -> dict:
        '''
        load_yaml function to load yaml file

        Args:
            fpath (str): path to the yaml file

        Returns:
            dict: yaml file as a dictionary
        '''
        with open(fpath, 'r') as f:
            data = yaml.load(f, Loader=yaml.FullLoader)
        f.close()

        return data

    @staticmethod 
    def save_pickle(file: list, path: str) -> None:
        '''
        save_pickle function to save list as pickle File

        Args:
            file (list): list to be saved
            path (str): path to the pickle File

        Returns:
            None
        '''
        with open(path, 'wb') as f:
            pickle.dump(file, f)
        f.close()

    @staticmethod
    def load_pickle(path: str) -> list:
        '''
        load_pickle function to load pickle File

        Args:
            path (str): path to the pickle File

        Returns:
            list: pickle File as a list
        '''
        with open(path, 'rb') as f:
            data = pickle.load(f)
        f.close()

        return data


class TomiUtils():

    @staticmethod
    def question_to_narrative(question: str) -> str:
        if 'really' in question:
            matched = re.match(r'Where is the ([a-z]*) really?', question)
            eoi = matched.group(1)
            new_narrative = f"At the end of the story, the {eoi} is located at "


class BaselineLabels():

    @property
    def fullness_labels(self) -> list[str]:
        return ['less full', 'equally full', 'more full']

    @property
    def weight_labels(self) -> list[str]:
        return ['lighter', 'equally heavy', 'heavier']

    @property
    def accessibility_labels(self) -> list[str]:
        return ['directly accessible', 'sealed in a container']
    
import os
import numpy as np
from glob import glob


class OpenToMUtils:


    def get_info(self, val: dict) -> tuple[str, str, str, str, str]:
        """
        function to get the characters, objects and locations involved in the ToMi narrative

        Args:
            val: a ToMi narrative entry

        Returns:
            mover: the character who moves the object 
            affected_char: the character who is potentially affected by the movement 
            original_place: the original location of the object 
            move_to_place: the destination location of the object 
            eoi: the object 
        """

        if 'plot_info' in val.keys():
            mover, affected_char, eoi, original_place, move_to_place = val['plot_info'].values()

        else:
            cur_content = val['plot']
            cur_questions = val['questions']
            all_context_ent = val['context_ent']

            eoi, coi = self.get_entity_of_interest(cur_questions, all_context_ent)

            content_sents = cur_content.split('\n')

            mover = ''
            move_to_place = ''
            original_place = ''
            flag = 1

            for sent in content_sents:

                if flag and eoi in sent:
                    sent_tokens = sent.replace('.', '').split() 
                    original_place = ''
                    for token in sent_tokens:
                        if token in all_context_ent and token != eoi and token[0].islower():
                            original_place += token
                            flag = 0

                if 'move' in sent:
                    sent_tokens = sent.replace('.', '').split()
                    mover = []
                    move_to_place = ''
                    for token in sent_tokens:
                        if token[0].isupper():
                            mover.append(token)

                    move_to_place = sent.split('to the')[-1].strip()

            # sanity check: there should be only one mover in the context
            mover = list(set(mover))
            if len(mover) > 1:
                raise ValueError('More than one mover found in the context.')

            mover = mover[0]
            # the mover should be in the characters of interest
            assert mover in coi, 'Mover not in characters of interest.'
            affected_char = [c for c in coi if c != mover]
            # there should only be one character affected in the context
            assert len(affected_char) == 1
            affected_char = affected_char[0]

            # there must be a place affected in the narrative
            assert move_to_place != '', 'No place affected found in the context.'

            # there must be an original place in the narrative
            # assert original_place != '', 'No original place found in the context.'

        return mover, affected_char, original_place, move_to_place, eoi


    @staticmethod
    def get_entity_of_interest(questions: dict, all_ents: list) -> tuple:
        """
        get_entity_of_interest funtion to get entity of interest in the questions. Returns the most common entity of interest.

        Args:
            questions: list of questions
            all_ents: list of all entities in the context

        Returns:
            str: object of interest
            list: characters of interest
        """
        eoi = None
        coi = []
        for ent in all_ents:
            if ent[0].islower() and ent in questions['1']['question']:
                eoi = ent

            for question in questions.values():
                if ent[0].isupper() and ent in question['question']:
                    coi.append(ent)

        if not eoi:
            raise ValueError('No entity of interest found in the context.')

        coi = list(set(coi))

        return (eoi, coi)


    @staticmethod 
    def cache_tom_data(data: dict, cache_path: str, model: str, **kwargs) -> None:
        datautils = DataUtils()
        existing_files = glob(os.path.join(cache_path, '*.json'))

        post_fix = ''
        for key, val in kwargs.items():
            if isinstance(val, str) and 'shot' in val:
                post_fix += '_' + f'{str(val)}_shot'
            elif val:
                post_fix += '_' + key.strip()

        existing_files = [file for file in existing_files if post_fix in file]
        existing_ids = [f.split('_')[-1].split('.')[0] for f in existing_files]
        existing_ids = [int(ele) for ele in existing_ids if ele.isnumeric()]

        new_id = np.random.randint(1000000, 9999999)
        while new_id in existing_ids:
            new_id = np.random.randint(1000000, 9999999)

        if model:
            new_fname = f'tomi_{model}' + post_fix + '_' + str(new_id) + '.json'
        else:
            new_fname = f'tomi' + post_fix + '_' + str(new_id) + '.json'

        datautils.save_json(data, os.path.join(cache_path, new_fname))
        
        
